In [1]:
import mrcfile
import numpy as np
from scipy.ndimage import binary_dilation, generate_binary_structure, binary_opening, binary_erosion, binary_closing
from scipy.ndimage import label
from scipy.ndimage import measurements
import os

In [2]:
working_dir = '/media/liushuo/data1/data/fig_demo_2/pp199/synapse_seg/'
base_name = 'pp199'
ori_tomo_path = working_dir + base_name + '/' + base_name + '.mrc'
bin2_tomo_path = working_dir + base_name + '/'   + 'synapse/' + base_name + '_bin2.mrc'
bin4_tomo_path = working_dir + base_name + '/'   + 'synapse/' + base_name + '_bin4.mrc'
bin4_synapse_label_path = working_dir + base_name + '/'  + 'synapse/' + base_name + '_bin4_label.mrc'
synapse_label_path = working_dir + base_name + '/'  + 'synapse/' + base_name + '_label.mrc'

prediction_label_path = working_dir + base_name + '/' + 'ret1.mrc'
save_mask_path = working_dir + base_name + '/' + 'membrane/' + base_name + '_membrane_label.mrc'
roi_label_path = working_dir + base_name + '/' + 'ret1_crop.mrc'

In [3]:
def get_tomo(path):
    """
    Load a 3D MRC file as a numpy array.

    Parameters:
    - path: str
        Path to the MRC file.

    Returns:
    - data: ndarray
        The 3D data loaded from the MRC file.
    """
    with mrcfile.open(path) as mrc:
        data = mrc.data
    return data

def save_tomo(data, path, voxel_size=17.14,datetype=np.int8):
    """
    Save a 3D numpy array as an MRC file.

    Parameters:
    - data: ndarray
        The 3D data to save.
    - voxel_size: float
        The voxel size of the data.
    """
        # 获取文件所在的目录
    directory = os.path.dirname(path)
    
    # 如果目录不存在，则创建
    if not os.path.exists(directory):
        os.makedirs(directory)
    
    with mrcfile.new(path, overwrite=True) as mrc:
        data = data.astype(datetype)
        mrc.set_data(data)
        mrc.voxel_size = voxel_size
        
def binarize_mask(mask: np.ndarray) -> np.ndarray:
    """
    将输入的三维mask数组进行01化处理，原本为0的不变，大于0的置为1。
    
    参数:
    mask (np.ndarray): 输入的三维np数组，代表实例分割的像素级mask。
    
    返回:
    np.ndarray: 处理后的二值化mask数组。
    """
    return np.where(mask > 0, 1, 0)

In [4]:

result_label = get_tomo(prediction_label_path)
prepost_mask = get_tomo(synapse_label_path)

In [5]:
# 对每个mask进行二值化处理
prepost_mask_bin = binarize_mask(prepost_mask)

In [6]:
dilation_radius = 6

def create_spherical_structure(radius):
    """
    创建一个球形的结构元，用于膨胀操作。

    参数:
    radius (int): 球的半径。

    返回:
    np.ndarray: 球形结构元。
    """
    # 生成球体网格
    size = 2 * radius + 1  # 球形结构元的大小
    z, y, x = np.ogrid[-radius:radius+1, -radius:radius+1, -radius:radius+1]
    distance = x**2 + y**2 + z**2
    # 将距离球心半径以内的点设为1，形成球形结构元
    spherical_structure = distance <= radius**2
    return spherical_structure

# 创建球形结构元
spherical_structure = create_spherical_structure(dilation_radius)  # 生成一个3D的结构元
# 对合并的mask进行球形膨胀
dilated_mask = binary_dilation(prepost_mask_bin, structure=spherical_structure)

roi_label = np.zeros_like(result_label)
final_label = np.zeros_like(result_label)

roi_label[dilated_mask] = result_label[dilated_mask]

# extract memb where roi_label == 5
final_label[roi_label == 5] = 1

# save_tomo(final_label, save_mask_path)

In [7]:
save_tomo(roi_label, roi_label_path)